<a href="https://colab.research.google.com/github/engogola/JusticeKE-RAG/blob/Master/JusticeKE_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

JusticeKE-RAG Is a specialized AI assistant focused on providing legal information, specifically related to Kenyan law

The system works by first attempting to find relevant answers within the documents  provided, such as the Kenyan Constitution and the Supreme Court Act and rules. This is achieved by extracting text from these documents, dividing it into manageable chunks, and then using embedding models and a vector store (FAISS) to create a searchable database.

If the required information isn't sufficiently found within these local documents, JusticeKE is designed to expand its search to the web, specifically targeting reliable sources like the kenyalaw.org website for constitutional articles.

Finally, the retrieved information, whether from local documents or web searches, is used to construct a prompt for the Gemini language model, which then generates the final answer to the user's legal question. Therefore, JusticeKE represents the complete workflow and infrastructure you've set up in your Colab notebook to power this legal question-answering system.

Install necesary dependencies and python libraries

In [ ]:
!pip install google-generativeai langchain faiss-cpu PyMuPDF
!pip install langchain_community


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.7 MB/s eta 0:00:00


This cell attempts to configure the Google Generative AI library, initially using a hardcoded API key placeholder.

In [ ]:
import google.generativeai as genai

genai.configure(api_key="GOOGLE_API_KEY")  # Use secrets or Colab env vars


This cell improves the API key handling by retrieving the Google API key from Colab secrets and then configuring the google.generativeai library and listing available models.

In [ ]:
import google.generativeai as genai
from google.colab import userdata # Import userdata to access Colab secrets

# Get the API key from Colab secrets
# Use the key name exactly as you defined it in the Colab secrets pane
try:
    api_key = userdata.get('GOOGLE_API_KEY')
except Exception as e:
    api_key = None
    print(f"Error accessing Colab secret: {e}")

# Check if the API key is loaded and configure genai
if api_key:
    genai.configure(api_key=api_key)  # Use the retrieved API key
    print("GOOGLE_API_KEY found. Configuring Generative AI.")

    # Find a suitable text generation model
    text_generation_model_name = None
    print("\nAvailable models:")
    for m in genai.list_models():
        print(f"- {m.name}")
        # Check if the model name indicates it's a text generation model
        # We'll look for names containing 'gemini' as a common indicator.
        # This is a heuristic and might need adjustment based on future API updates.
        if 'gemini' in m.name:
             print(f"  (Supports text generation - heuristic based on name)")
             if text_generation_model_name is None:
                text_generation_model_name = m.name # Store the first suitable model name

    if text_generation_model_name:
        print(f"\nFound a text generation model: {text_generation_model_name}")
        # Initialize the text generation model
        model = genai.GenerativeModel(text_generation_model_name)
        print(f"Initialized model: {model.model_name}")
        # You can now use the 'model' object to generate text
        # For example:
        # response = model.generate_content("Tell me a story.")
        # print(response.text)
    else:
        print("\nNo text generation model found among the available models based on naming heuristic.")
        print("Please check the available models listed above and manually specify a suitable model name if needed.")


else:
    print("Error: GOOGLE_API_KEY not found in Colab secrets or an error occurred.")
    print("Please go to the key icon on the left sidebar, click 'Add new secret',")
    print("and add a secret with the name 'GOOGLE_API_KEY' and your API key as the value.")

**PDF Text Extraction**

This cell defines a function to extract text from PDF files and then uses it to read the content of the Kenyan Constitution and Supreme Court Act PDFs.

In [ ]:
import fitz  # PyMuPDF

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

# Example usage
kenyan_constitution_text = extract_text_from_pdf("/content/Kenyan Constitution.pdf")
supreme_court_text = extract_text_from_pdf("/content/SupremeCourtActrules-of-2020.pdf")


**Text Chunking**

This cell defines a function to break down large text documents into smaller, overlapping chunks.

In [ ]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(kenyan_constitution_text)


**Langchain Google GenAI Installation**

This cell installs the Langchain integration for Google's Generative AI models.

In [ ]:
!pip install langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.1 MB/s eta 0:00:00
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.6.15
    Uninstalling google-ai-generativelanguage-0.6.15:
      Successfully uninstalled google-ai-generativelanguage-0.6.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.18 which is incompatible.


**Embeddings and FAISS Database Creation:**

This cell initializes a Google Generative AI embedding model using your API key and then creates a searchable FAISS vector database from the text chunks extracted from the documents.

In [ ]:
# Update the import statement
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document
from langchain_community.vectorstores import FAISS # Import FAISS from langchain_community
import os # Import the os module to access environment variables
from google.colab import userdata # Import userdata to access Colab secrets

# Initialize the embedding model, specifying the task type
# Ensure you have set the GOOGLE_API_KEY in Colab secrets

# Get the API key from Colab secrets
# Use the key name exactly as you defined it in the Colab secrets pane
try:
    api_key = userdata.get('GOOGLE_API_KEY')
except Exception as e:
    api_key = None
    print(f"Error accessing Colab secret: {e}")


# Add a check to see if the API key is loaded
if not api_key:
    print("Error: GOOGLE_API_KEY not found in Colab secrets or an error occurred.")
    print("Please go to the key icon on the left sidebar, click 'Add new secret',")
    print("and add a secret with the name 'GOOGLE_API_KEY' and your API key as the value.")
else:
    print("GOOGLE_API_KEY found. Initializing embedding model.")
    # Based on Google's documentation [1], SEMANTIC_SIMILARITY is a common task type.
    # Pass the API key explicitly to the constructor
    embedding = GoogleGenerativeAIEmbeddings(
        model="models/embedding-001",
        task_type="SEMANTIC_SIMILARITY",
        google_api_key=api_key # Use the retrieved API key
    )

    docs = [Document(page_content=chunk) for chunk in chunks]
    # Use the FAISS class imported from langchain_community
    db = FAISS.from_documents(docs, embedding)
    print("FAISS database created successfully.")

GOOGLE_API_KEY found. Initializing embedding model.
FAISS database created successfully.


**Retriever Setup**

This cell creates a retriever object from the FAISS database, configured to fetch the top 3 relevant documents based on a query.

In [ ]:
retriever = db.as_retriever(search_kwargs={"k": 3})

def retrieve_context(query):
    return retriever.get_relevant_documents(query)



**Context Retrieval **

This cell defines a function that uses the retriever to get relevant document chunks based on a user query.

In [ ]:
retrieved_docs = retriever.get_relevant_documents(query)
context = "\n\n".join([doc.page_content for doc in retrieved_docs])


<ipython-input-18-0112879f0886>:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retrieved_docs = retriever.get_relevant_documents(query)



 **Prompt Construction (Basic):**

This cell shows how to build a basic prompt for the Gemini model using the retrieved context and a placeholder user query.

In [ ]:
prompt = f"""
You are a Kenyan legal assistant. Use the following legal context to answer the user's question accurately and in a grounded way.

Context:
{context}

User Question:
{query}

Answer:
"""


In [ ]:
import google.generativeai as genai

model = genai.GenerativeModel("gemini-1.5-flash-latest")
response = model.generate_content(prompt)
print(response.text)


To answer your question accurately, I need to refer to more than just the chapter headings provided.  The provided text only lists chapter headings from the Constitution of Kenya, 2010,  referencing articles related to fair hearing (Article 50) and the rights of persons detained, held in custody, or imprisoned (Article 51).  These articles themselves contain the specifics of the rights of arrested persons.  

Therefore, I cannot provide a complete answer based solely on the given information.  To understand the rights of arrested persons in Kenya, you must consult the full text of Articles 50 and 51 of the Constitution of Kenya, 2010, and related legislation.  These articles will detail rights such as the right to a fair trial, the right to remain silent, the right to legal representation, and the right to be informed of the charges against them.  Furthermore, other articles within the Constitution may also apply, depending on the specific circumstances of the arrest (e.g., Articles 53


Full RAG Workflow (with Fallback)

This cell implements the main RAG (Retrieval Augmented Generation) workflow, attempting to retrieve information from the local database first, and falling back to a simulated web search if local retrieval is insufficient, before calling the Gemini model.

In [ ]:
# STEP 1: User types a question
user_query = input("Ask your legal question: ")

# Define a placeholder search_google function.
# In a real scenario, you would implement this function
# using a search API (like Google Search API or SerpApi)
# to fetch relevant information from the web.
def search_google(query):
    print(f"(Simulating Google Search for: {query})")
    # This is a placeholder. Replace with actual search API call.
    # For demonstration, returning a generic message.
    return f"Information about '{query}' from a web search."

# STEP 2: Try retrieving from attached documents (via FAISS or ChromaDB)
# Ensure 'retriever' is defined from a previous cell
try:
    retrieved_docs = retriever.get_relevant_documents(user_query)
except NameError:
    print("Error: 'retriever' is not defined. Please run the cell that initializes the FAISS database.")
    retrieved_docs = [] # Assign empty list to prevent further errors


if retrieved_docs and len(retrieved_docs) > 0:
    # Context found from your own documents
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    prompt = f"""
You are a Kenyan legal assistant AI. Based only on the documents below, answer the question accurately.

Context:
{context}

User Question:
{user_query}

Answer:
"""
else:
    # Not enough context — fallback to external web search
    print("⚠️ Not enough info found locally. Searching online...")

    # Ensure search_google function is defined
    article_50 = search_google("Article 50 Constitution of Kenya site:kenyalaw.org")
    article_51 = search_google("Article 51 Constitution of Kenya site:kenyalaw.org")

    prompt = f"""
The user wants to know their legal rights under the Kenyan Constitution. Here is some online information:

Article 50: {article_50}
Article 51: {article_51}

User Question:
{user_query}

Answer:
"""

# STEP 3: Call Gemini with the constructed prompt
# Ensure 'model' is defined from a previous cell where the GenerativeModel was initialized
from IPython import get_ipython
from IPython.display import display

try:
    response = model.generate_content(prompt)
    print("\n✅ Response:\n")

    # Format the response text in markdown
    # This is a basic example; you might need more sophisticated parsing
# STEP 1: User types a question
user_query = input("Ask your legal question: ")

# Define a placeholder search_google function.
# In a real scenario, you would implement this function
# using a search API (like Google Search API or SerpApi)
# to fetch relevant information from the web.
def search_google(query):
    print(f"(Simulating Google Search for: {query})")
    # This is a placeholder. Replace with actual search API call.
    # For demonstration, returning a generic message.
    return f"Information about '{query}' from a web search."

# STEP 2: Try retrieving from attached documents (via FAISS or ChromaDB)
# Ensure 'retriever' is defined from a previous cell
try:
    retrieved_docs = retriever.get_relevant_documents(user_query)
except NameError:
    print("Error: 'retriever' is not defined. Please run the cell that initializes the FAISS database.")
    retrieved_docs = [] # Assign empty list to prevent further errors


if retrieved_docs and len(retrieved_docs) > 0:
    # Context found from your own documents
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    prompt = f"""
You are a Kenyan legal assistant AI. Based only on the documents below, answer the question accurately.

Context:
{context}

User Question:
{user_query}

Answer:
"""
else:
    # Not enough context — fallback to external web search
    print("⚠️ Not enough info found locally. Searching online...")

    # Ensure search_google function is defined
    article_50 = search_google("Article 50 Constitution of Kenya site:kenyalaw.org")
    article_51 = search_google("Article 51 Constitution of Kenya site:kenyalaw.org")

    prompt = f"""
The user wants to know their legal rights under the Kenyan Constitution. Here is some online information:

Article 50: {article_50}
Article 51: {article_51}

User Question:
{user_query}

Answer:
"""

# STEP 3: Call Gemini with the constructed prompt
# Ensure 'model' is defined from a previous cell where the GenerativeModel was initialized
from IPython import get_ipython
from IPython.display import display

try:
    response = model.generate_content(prompt)
    print("\n✅ Response:\n")

    # Format the response text in markdown
    # This is a basic example; you might need more sophisticated parsing
    # depending on the structure of the generated text.
    formatted_response = response.text.replace("\n", "\n\n") # Add extra newline for spacing
    formatted_response = f"{formatted_response}"
    display(formatted_response) # Display formatted text

except NameError:
    print("Error: 'model' is not defined. Please run the cell that initializes the GenerativeModel.")
except Exception as e:
    print(f"An error occurred during model generation: {e}")

SyntaxError: unterminated string literal (detected at line 71) (<ipython-input-1-36a8bfd8ec14>, line 71)